# IEP-1002 Day 3 — RAGAS 4지표 평가

구조 기반 청킹(`ipcc_structural_v1`, 284청크)을 IEP-1001 CASE 3 대비 4지표로 평가합니다.

| 지표 | 구분 | IEP-1001 CASE3 baseline |
| :--- | :--- | :---: |
| Context Recall | Retriever | 0.8520 |
| Context Precision | Retriever | 미측정 |
| Faithfulness | Generator | 미측정 |
| Answer Relevancy | Generator | 미측정 |

**평가 전략**: T4 메모리 제약으로 2회 분리 실행  
**전제 조건**: IEP-1002 Day 2 산출물(`chroma_db/`, `ipcc_structural_v1`)이 Drive에 존재해야 합니다.

## Step 0 — Config

In [ ]:
DRIVE_BASE   = "/content/drive/MyDrive/IPCC_data/IEP_1002"
CHROMA_DIR   = f"{DRIVE_BASE}/chroma_db"
GOLDEN_CSV   = f"{DRIVE_BASE}/golden_dataset_100.csv"
RESULTS_DIR  = f"{DRIVE_BASE}/results"

COLLECTION_NAME = "ipcc_structural_v1"
EMBED_MODEL     = "jhgan/ko-sroberta-multitask"
JUDGE_MODEL     = "llama3.1:8b"
TOP_K           = 3

SAVE_RETRIEVED = f"{RESULTS_DIR}/iep1002_day3_retrieved.csv"
SAVE_RETRIEVER = f"{RESULTS_DIR}/iep1002_day3_retriever.csv"
SAVE_GENERATOR = f"{RESULTS_DIR}/iep1002_day3_generator.csv"
SAVE_RAW       = f"{RESULTS_DIR}/iep1002_day3_raw.csv"
SAVE_SUMMARY   = f"{RESULTS_DIR}/iep1002_day3_summary.csv"

BASELINE = {
    "전체":    {"context_recall": 0.8520},
    "사실 확인": {"context_recall": 0.8627},
    "비교":    {"context_recall": 0.8897},
    "의견/예측": {"context_recall": 0.8875},
    "범위 밖":  {"context_recall": 0.8595},
}

print("✅ Config 완료")

## Step 1 — 라이브러리 설치

In [ ]:
!pip install -qU chromadb langchain-community langchain-huggingface langchain-ollama ragas tqdm
print("✅ 설치 완료")

## Step 2 — Drive 마운트 + Ollama 서버 시작

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs(RESULTS_DIR, exist_ok=True)
print("✅ Drive 마운트 완료")

In [ ]:
import subprocess, time, requests

def start_ollama():
    subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for i in range(30):
        try:
            if requests.get("http://localhost:11434", timeout=2).status_code == 200:
                print(f"✅ Ollama 서버 준비 완료 ({i+1}초)")
                return True
        except:
            pass
        time.sleep(1)
    print("❌ 서버 시작 실패")
    return False

def check_ollama():
    try:
        requests.get("http://localhost:11434", timeout=2)
        print("✅ Ollama 서버 정상")
        return True
    except:
        print("❌ 서버 응답 없음 — start_ollama()를 다시 실행하세요.")
        return False

!curl -fsSL https://ollama.com/install.sh | sh 2>/dev/null
start_ollama()
!ollama pull {JUDGE_MODEL}
!ollama pull nomic-embed-text
print("✅ 모델 준비 완료")

## Step 3 — ChromaDB + 임베딩 모델 로드

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)
count = vectorstore._collection.count()
print(f"✅ ChromaDB 로드: {COLLECTION_NAME} ({count}개 청크)")
assert count > 0, "⚠️  청크 0개 — CHROMA_DIR 경로를 확인하세요."

## Step 4 — 골든 데이터셋 로드

In [ ]:
import pandas as pd
df_gold = pd.read_csv(GOLDEN_CSV)
print(f"✅ {len(df_gold)}개 로드")
print(df_gold['Type'].value_counts().to_string())

## Step 5 — Retrieval (top-3)

In [ ]:
from tqdm.notebook import tqdm

retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
records = []
for _, row in tqdm(df_gold.iterrows(), total=len(df_gold), desc="Retrieving"):
    docs = retriever.invoke(row['user_input'])
    records.append({
        "id":                 row['ID'],
        "type":               row['Type'],
        "user_input":         row['user_input'],
        "reference":          row['reference'],
        "retrieved_contexts": [doc.page_content for doc in docs],
    })
print(f"✅ Retrieval 완료: {len(records)}개")

## Step 6 — Answer 생성 (RAG)

In [ ]:
import requests as req

RAG_PROMPT = """다음 문서를 참고하여 질문에 답하세요.
문서에 없는 내용은 "해당 내용은 보고서에서 찾을 수 없습니다."라고 답하세요.

[문서]
{context}

[질문]
{question}

[답변]"""

def generate_answer(question, contexts):
    context = "\n\n".join(f"[문서 {i+1}]\n{c}" for i, c in enumerate(contexts))
    try:
        resp = req.post(
            "http://localhost:11434/api/generate",
            json={"model": JUDGE_MODEL, "prompt": RAG_PROMPT.format(context=context, question=question), "stream": False},
            timeout=120
        )
        return resp.json().get("response", "").strip()
    except Exception as e:
        print(f"  ⚠️  생성 실패: {e}")
        return ""

for rec in tqdm(records, desc="Generating answers"):
    rec['answer'] = generate_answer(rec['user_input'], rec['retrieved_contexts'])

empty = sum(1 for r in records if not r['answer'])
print(f"✅ 답변 생성 완료 (실패: {empty}개 / 전체: {len(records)}개)")

## Step 7 — 중간 저장

> 세션이 끊겨도 Step 8부터 재개할 수 있습니다.

In [ ]:
import json
pd.DataFrame([{
    "id": r['id'], "type": r['type'], "user_input": r['user_input'],
    "reference": r['reference'],
    "retrieved_contexts": json.dumps(r['retrieved_contexts'], ensure_ascii=False),
    "answer": r['answer'],
} for r in records]).to_csv(SAVE_RETRIEVED, index=False, encoding='utf-8-sig')
print(f"✅ 중간 저장: {SAVE_RETRIEVED}")

In [ ]:
# ── 세션 재시작 후 복원 (Step 0 먼저 실행) ───────────────────────
# import json, pandas as pd
# df_ = pd.read_csv(SAVE_RETRIEVED, encoding='utf-8-sig')
# records = [{
#     "id": r['id'], "type": r['type'], "user_input": r['user_input'],
#     "reference": r['reference'],
#     "retrieved_contexts": json.loads(r['retrieved_contexts']),
#     "answer": r['answer'],
# } for _, r in df_.iterrows()]
# print(f"✅ 복원 완료: {len(records)}개")

## Step 8 — RAGAS 평가 모델 설정

In [ ]:
check_ollama()

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from ragas import EvaluationDataset

ragas_llm = LangchainLLMWrapper(
    ChatOllama(model=JUDGE_MODEL, base_url="http://localhost:11434", temperature=0, num_predict=512)
)
ragas_embeddings = LangchainEmbeddingsWrapper(
    OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")
)
run_config = RunConfig(max_workers=1, timeout=600, max_retries=10)

eval_dataset = EvaluationDataset.from_pandas(pd.DataFrame([{
    "user_input":         r['user_input'],
    "retrieved_contexts": r['retrieved_contexts'],
    "response":           r['answer'],
    "reference":          r['reference'],
} for r in records]))

print(f"✅ 평가 모델 설정 완료 / EvaluationDataset: {len(eval_dataset)}개")

## Step 9 — 1회차 평가: Context Recall + Context Precision

In [ ]:
from ragas import evaluate
from ragas.metrics import ContextRecall, ContextPrecision

print(f"▶ 1회차 평가 시작 (judge: {JUDGE_MODEL})")
result_retriever = evaluate(
    dataset=eval_dataset,
    metrics=[ContextRecall(), ContextPrecision()],
    llm=ragas_llm, embeddings=ragas_embeddings, run_config=run_config,
)
df_retriever = result_retriever.to_pandas()
df_retriever['id']   = [r['id']   for r in records]
df_retriever['type'] = [r['type'] for r in records]
df_retriever.to_csv(SAVE_RETRIEVER, index=False, encoding='utf-8-sig')

print("\n✅ 1회차 완료")
print(df_retriever[['context_recall','context_precision']].mean().round(4).to_string())
print("NaN:", df_retriever[['context_recall','context_precision']].isna().sum().to_dict())

## Step 10 — 2회차 평가: Faithfulness + Answer Relevancy

In [ ]:
check_ollama()

In [ ]:
from ragas.metrics import Faithfulness, AnswerRelevancy

print(f"▶ 2회차 평가 시작 (judge: {JUDGE_MODEL})")
result_generator = evaluate(
    dataset=eval_dataset,
    metrics=[Faithfulness(), AnswerRelevancy()],
    llm=ragas_llm, embeddings=ragas_embeddings, run_config=run_config,
)
df_generator = result_generator.to_pandas()
df_generator['id']   = [r['id']   for r in records]
df_generator['type'] = [r['type'] for r in records]
df_generator.to_csv(SAVE_GENERATOR, index=False, encoding='utf-8-sig')

print("\n✅ 2회차 완료")
print(df_generator[['faithfulness','answer_relevancy']].mean().round(4).to_string())
print("NaN:", df_generator[['faithfulness','answer_relevancy']].isna().sum().to_dict())

## Step 11 — 결과 병합 + 유형별 분석

In [ ]:
import numpy as np

METRICS = ['context_recall', 'context_precision', 'faithfulness', 'answer_relevancy']
df_raw = pd.DataFrame({
    'id':                [r['id']   for r in records],
    'type':              [r['type'] for r in records],
    'user_input':        [r['user_input'] for r in records],
    'context_recall':    df_retriever['context_recall'].values,
    'context_precision': df_retriever['context_precision'].values,
    'faithfulness':      df_generator['faithfulness'].values,
    'answer_relevancy':  df_generator['answer_relevancy'].values,
})
overall_mean = df_raw[METRICS].mean().round(4)
overall_nan  = df_raw[METRICS].isna().sum()
summary      = df_raw.groupby('type')[METRICS].mean().round(4)
nan_by_type  = df_raw.groupby('type')[METRICS].apply(lambda x: x.isna().sum())

print("전체 평균 (NaN 제외):")
for m in METRICS:
    print(f"  {m:<22}: {overall_mean[m]:.4f}  (NaN: {overall_nan[m]}개)")
print("\n유형별 평균:")
print(summary.to_string())
print("\n유형별 NaN:")
print(nan_by_type.to_string())

## Step 12 — IEP-1001 비교 + 최종 저장

In [ ]:
types_order = ['사실 확인', '비교', '의견/예측', '범위 밖']

print("Context Recall 비교: IEP-1001 CASE3 vs IEP-1002")
print(f"  {'유형':<10} {'IEP-1001 CASE3':>16} {'IEP-1002':>12} {'차이':>8}")
print("-" * 55)
for t in types_order:
    b = BASELINE.get(t, {}).get('context_recall', np.nan)
    n = summary.loc[t, 'context_recall'] if t in summary.index else np.nan
    diff = n - b if not (np.isnan(n) or np.isnan(b)) else np.nan
    print(f"  {t:<10} {b:>16.4f} {n:>12.4f} {f'{diff:+.4f}':>8}")
b_all, n_all = BASELINE['전체']['context_recall'], overall_mean['context_recall']
print("-" * 55)
print(f"  {'전체':<10} {b_all:>16.4f} {n_all:>12.4f} {f'{n_all-b_all:+.4f}':>8}")

print("\n[IEP 문서 마크다운]")
md = "| 유형 | Context Recall | Context Precision | Faithfulness | Answer Relevancy |\n"
md += "| :--- | :---: | :---: | :---: | :---: |\n"
for t in types_order:
    if t in summary.index:
        r = summary.loc[t]
        md += f"| {t} | {r['context_recall']:.4f} | {r['context_precision']:.4f} | {r['faithfulness']:.4f} | {r['answer_relevancy']:.4f} |\n"
md += f"| **전체** | **{overall_mean['context_recall']:.4f}** | **{overall_mean['context_precision']:.4f}** | **{overall_mean['faithfulness']:.4f}** | **{overall_mean['answer_relevancy']:.4f}** |\n"
print(md)

df_raw.to_csv(SAVE_RAW, index=False, encoding='utf-8-sig')
df_s = summary.copy()
df_s.loc['전체'] = overall_mean
df_s.to_csv(SAVE_SUMMARY, encoding='utf-8-sig')
print(f"✅ 저장 완료 → {SAVE_RAW}")